In [1]:
import numpy as np
import rasterio
from pathlib import Path
import geopandas as gpd
from rasterio.mask import mask
import json
import calendar

# =========================================================
# PATHS
# =========================================================

ROOT = Path(
    "/home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/data_extractor/era5_land"
)

quarter_dir = ROOT / "data" / "quarterly_tiffs"
threshold_dir = ROOT / "data" / "percentile_tiffs"
output_dir = ROOT / "data" / "monthly_heatday_scores"

geojson_path = Path(
    "/home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/data_extractor/assets/district.geojson"
)

output_dir.mkdir(parents=True, exist_ok=True)

# =========================================================
# LOAD DISTRICT GEOMETRY
# =========================================================

gdf = gpd.read_file(geojson_path)

geoms = json.loads(
    gdf.to_json()
)["features"]

geoms = [
    f["geometry"]
    for f in geoms
]

# =========================================================
# MONTH NAMES
# =========================================================

month_names = {
    1: "JAN",
    2: "FEB",
    3: "MAR",
    4: "APR",
    5: "MAY",
    6: "JUN",
    7: "JUL",
    8: "AUG",
    9: "SEP",
    10: "OCT",
    11: "NOV",
    12: "DEC",
}

# =========================================================
# LOAD SINGLE BAND RASTER
# =========================================================

def load_raster(path):

    with rasterio.open(path) as src:

        return src.read(1)

# =========================================================
# QUARTER → MONTH BAND OFFSETS
# =========================================================

def get_band_slice_for_month(
    year: int,
    month: int
):
    """
    Returns 1-based band indices
    for the requested month
    within a quarterly TIFF.
    """

    quarter_first_month = (
        ((month - 1) // 3) * 3 + 1
    )

    day_offset = 0

    for m in range(
        quarter_first_month,
        month
    ):

        day_offset += (
            calendar.monthrange(
                year,
                m
            )[1]
        )

    days_in_month = (
        calendar.monthrange(
            year,
            month
        )[1]
    )

    band_start = day_offset + 1
    band_end = (
        day_offset +
        days_in_month
    )

    return (
        band_start,
        band_end
    )

# =========================================================
# YEARS
# =========================================================

years = [
    2021,
    2022,
    2023,
    2024,
    2025,
    2026
]

# =========================================================
# MAIN LOOP
# =========================================================

for year in years:

    print(
        f"\n================ YEAR {year} ================"
    )

    for month in range(1, 13):

        month_name = month_names[month]

        print(
            f"\nProcessing {year}-{month:02d}"
        )

        # -------------------------------------------------
        # QUARTER MAPPING
        # -------------------------------------------------

        if month in [1, 2, 3]:
            q = "Q1"

        elif month in [4, 5, 6]:
            q = "Q2"

        elif month in [7, 8, 9]:
            q = "Q3"

        else:
            q = "Q4"

        hi_path = (
            quarter_dir /
            f"HI_{year}_{q}.tif"
        )

        if not hi_path.exists():

            print(
                f"SKIP: missing {hi_path.name}"
            )

            continue

        # -------------------------------------------------
        # LOAD MONTH-SPECIFIC THRESHOLDS
        # -------------------------------------------------

        try:

            p80 = load_raster(
                threshold_dir /
                f"{month_name}_P80_1990_2023.tif"
            )

            p88 = load_raster(
                threshold_dir /
                f"{month_name}_P88_1990_2023.tif"
            )

            p95 = load_raster(
                threshold_dir /
                f"{month_name}_P95_1990_2023.tif"
            )

            p99 = load_raster(
                threshold_dir /
                f"{month_name}_P99_1990_2023.tif"
            )

        except Exception as e:

            print(
                f"SKIP: missing thresholds "
                f"for {month_name} — {e}"
            )

            continue

        # -------------------------------------------------
        # FIND MONTH BANDS
        # -------------------------------------------------

        band_start, band_end = (
            get_band_slice_for_month(
                year,
                month
            )
        )

        expected_days = (
            calendar.monthrange(
                year,
                month
            )[1]
        )

        print(
            f"Quarter {q}, "
            f"Bands {band_start}-{band_end} "
            f"({expected_days} days)"
        )

        # -------------------------------------------------
        # READ MONTH FROM QUARTERLY TIFF
        # -------------------------------------------------

        with rasterio.open(hi_path) as src:

            total_bands = src.count

            if band_end > total_bands:

                print(
                    f"WARN: raster has "
                    f"{total_bands} bands, "
                    f"requested {band_end}"
                )

                band_end = total_bands

            band_indices = list(
                range(
                    band_start,
                    band_end + 1
                )
            )

            hi = src.read(
                band_indices
            )

            meta = src.meta.copy()

        days, rows, cols = hi.shape

        print(
            f"Loaded {days} daily layers"
        )

        # -------------------------------------------------
        # HEATDAY SCORE
        # -------------------------------------------------

        monthly = np.zeros(
            (rows, cols),
            dtype=np.float32
        )

        for d in range(days):

            h = hi[d]

            score = np.zeros(
                (rows, cols),
                dtype=np.uint8
            )

            score[
                (h >= p80) &
                (h < p88)
            ] = 1

            score[
                (h >= p88) &
                (h < p95)
            ] = 2

            score[
                (h >= p95) &
                (h < p99)
            ] = 3

            score[
                h >= p99
            ] = 4

            monthly += score

        # -------------------------------------------------
        # TEMP FILE
        # -------------------------------------------------

        tmp_file = (
            output_dir /
            f"_tmp_{year}_{month:02d}.tif"
        )

        NODATA_VAL = -9999.0

        meta.update(
            {
                "count": 1,
                "dtype": "float32",
                "nodata": NODATA_VAL
            }
        )

        with rasterio.open(
            tmp_file,
            "w",
            **meta
        ) as dst:

            dst.write(
                monthly,
                1
            )

        # -------------------------------------------------
        # CLIP TO DISTRICT BOUNDARY
        # -------------------------------------------------

        with rasterio.open(
            tmp_file
        ) as src:

            out_image, out_transform = mask(
                src,
                geoms,
                crop=True,
                filled=True,
                nodata=NODATA_VAL,
                all_touched=True
            )

        out_image = out_image.astype(
            "float32"
        )

        out_final = out_image[0].copy()

        out_final[
            out_final == NODATA_VAL
        ] = np.nan

        out_meta = meta.copy()

        out_meta.update(
            {
                "height": out_image.shape[1],
                "width": out_image.shape[2],
                "transform": out_transform,
                "count": 1,
                "nodata": np.nan,
            }
        )

        # -------------------------------------------------
        # FINAL SAVE
        # -------------------------------------------------

        final_file = (
            output_dir /
            f"HEATDAY_{year}_{month:02d}.tif"
        )

        with rasterio.open(
            final_file,
            "w",
            **out_meta
        ) as dst:

            dst.write(
                out_final,
                1
            )

        valid_pixels = int(
            np.sum(
                ~np.isnan(out_final)
            )
        )

        max_score = (
            np.nanmax(out_final)
            if valid_pixels > 0
            else "N/A"
        )

        print(
            f"Saved: {final_file.name} | "
            f"valid={valid_pixels} | "
            f"max_score={max_score}"
        )

        tmp_file.unlink()

print("\n====================================")
print("MONTHLY HEATDAY PROCESSING COMPLETE")
print("====================================")


================ YEAR 2021 ================

Processing 2021-01
Quarter Q1, Bands 1-31 (31 days)
Loaded 31 daily layers
Saved: HEATDAY_2021_01.tif | valid=2266 | max_score=50.0

Processing 2021-02
Quarter Q1, Bands 32-59 (28 days)
Loaded 28 daily layers
Saved: HEATDAY_2021_02.tif | valid=2266 | max_score=13.0

Processing 2021-03
Quarter Q1, Bands 60-90 (31 days)
Loaded 31 daily layers
Saved: HEATDAY_2021_03.tif | valid=2266 | max_score=28.0

Processing 2021-04
Quarter Q2, Bands 1-30 (30 days)
Loaded 30 daily layers
Saved: HEATDAY_2021_04.tif | valid=2266 | max_score=7.0

Processing 2021-05
Quarter Q2, Bands 31-61 (31 days)
Loaded 31 daily layers
Saved: HEATDAY_2021_05.tif | valid=2266 | max_score=8.0

Processing 2021-06
Quarter Q2, Bands 62-91 (30 days)
Loaded 30 daily layers
Saved: HEATDAY_2021_06.tif | valid=2266 | max_score=8.0

Processing 2021-07
Quarter Q3, Bands 1-31 (31 days)
Loaded 31 daily layers
Saved: HEATDAY_2021_07.tif | valid=2266 | max_score=26.0

Processing 2021-08
Qua

In [ ]:
# summer heatday folders

In [6]:
# =========================================================
# SUMMER HEATDAY SCORE
# Q2 ONLY (APR–JUN)
#
# INPUT:
# quarterly_tiffs_summer2125/
# ├── HI_2021_Q2.tif
# ├── HI_2022_Q2.tif
# ├── HI_2023_Q2.tif
# ├── HI_2024_Q2.tif
# └── HI_2025_Q2.tif
#
# THRESHOLDS:
# AMJ_P80_1990_2023.tif
# AMJ_P88_1990_2023.tif
# AMJ_P95_1990_2023.tif
# AMJ_P99_1990_2023.tif
#
# OUTPUT:
# summer_heatday_scores/
# ├── SUMMER_HEATDAY_2021.tif
# ├── SUMMER_HEATDAY_2022.tif
# ├── SUMMER_HEATDAY_2023.tif
# ├── SUMMER_HEATDAY_2024.tif
# └── SUMMER_HEATDAY_2025.tif
#
# Each output pixel =
# sum of daily heatday scores over Apr-Jun
#
# Daily scoring:
# P80–P88 = 1
# P88–P95 = 2
# P95–P99 = 3
# >=P99   = 4
# =========================================================

import json
from pathlib import Path

import geopandas as gpd
import numpy as np
import rasterio
from rasterio.mask import mask

# =========================================================
# PATHS
# =========================================================

ROOT = Path(
    "/home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/data_extractor/era5_land"
)

quarter_dir = (
    ROOT
    / "data"
    / "quarterly_tiffs_summer2125"
)

threshold_dir = (
    ROOT
    / "data"
    / "percentile_tiffs"
)

output_dir = (
    ROOT
    / "data"
    / "summer_heatday_scores"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

geojson_path = Path(
    "/home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/data_extractor/assets/district.geojson"
)

# =========================================================
# LOAD DISTRICT BOUNDARY
# =========================================================

gdf = gpd.read_file(geojson_path)

geoms = json.loads(
    gdf.to_json()
)["features"]

geoms = [
    f["geometry"]
    for f in geoms
]

# =========================================================
# LOAD SINGLE BAND RASTER
# =========================================================

def load_raster(path):

    with rasterio.open(path) as src:
        return src.read(1)

# =========================================================
# LOAD AMJ THRESHOLDS
# =========================================================

print("\nLoading AMJ thresholds...")

p80 = load_raster(
    threshold_dir / "AMJ_P80_1990_2023.tif"
)

p88 = load_raster(
    threshold_dir / "AMJ_P88_1990_2023.tif"
)

p95 = load_raster(
    threshold_dir / "AMJ_P95_1990_2023.tif"
)

p99 = load_raster(
    threshold_dir / "AMJ_P99_1990_2023.tif"
)

print("Thresholds loaded.")

# =========================================================
# PROCESS YEARS
# =========================================================

years = [2021, 2022, 2023, 2024, 2025]

NODATA_VAL = -9999.0

for year in years:

    print(
        f"\n================ {year} ================"
    )

    hi_path = (
        quarter_dir
        / f"HI_{year}_Q2.tif"
    )

    if not hi_path.exists():

        print(
            f"Missing: {hi_path}"
        )

        continue

    # =====================================================
    # LOAD ALL DAILY BANDS
    # =====================================================

    with rasterio.open(hi_path) as src:

        hi = src.read()          # (days, rows, cols)

        meta = src.meta.copy()

        print(
            f"Loaded {src.count} daily layers"
        )

    days, rows, cols = hi.shape

    # =====================================================
    # COMPUTE SUMMER HEATDAY SCORE
    # =====================================================

    summer_score = np.zeros(
        (rows, cols),
        dtype=np.float32
    )

    for d in range(days):

        h = hi[d]

        score = np.zeros(
            (rows, cols),
            dtype=np.uint8
        )

        score[
            (h >= p80)
            & (h < p88)
        ] = 1

        score[
            (h >= p88)
            & (h < p95)
        ] = 2

        score[
            (h >= p95)
            & (h < p99)
        ] = 3

        score[
            h >= p99
        ] = 4

        summer_score += score

    print(
        f"Max score: {np.nanmax(summer_score):.1f}"
    )

    # =====================================================
    # WRITE TEMP FILE
    # =====================================================

    tmp_file = (
        output_dir
        / f"_tmp_{year}.tif"
    )

    meta.update(
        {
            "count": 1,
            "dtype": "float32",
            "nodata": NODATA_VAL
        }
    )

    with rasterio.open(
        tmp_file,
        "w",
        **meta
    ) as dst:

        dst.write(
            summer_score,
            1
        )

    # =====================================================
    # CLIP TO DISTRICT BOUNDARY
    # =====================================================

    with rasterio.open(tmp_file) as src:

        out_image, out_transform = mask(
            src,
            geoms,
            crop=True,
            filled=True,
            nodata=NODATA_VAL,
            all_touched=True
        )

    out_final = out_image[0].astype(
        np.float32
    )

    out_final[
        out_final == NODATA_VAL
    ] = np.nan

    # =====================================================
    # OUTPUT METADATA
    # =====================================================

    out_meta = meta.copy()

    out_meta.update(
        {
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform,
            "count": 1,
            "nodata": np.nan
        }
    )

    # =====================================================
    # SAVE FINAL RASTER
    # =====================================================

    out_file = (
        output_dir
        / f"SUMMER_HEATDAY_{year}.tif"
    )

    with rasterio.open(
        out_file,
        "w",
        **out_meta
    ) as dst:

        dst.write(
            out_final,
            1
        )

    tmp_file.unlink()

    valid_pixels = int(
        np.sum(
            ~np.isnan(out_final)
        )
    )

    print(
        f"Saved: {out_file.name}"
    )

    print(
        f"Valid pixels: {valid_pixels}"
    )

    print(
        f"Max score: "
        f"{np.nanmax(out_final):.1f}"
    )

print(
    "\n===================================="
)

print(
    "ALL SUMMER HEATDAY RASTERS CREATED"
)

print(
    "===================================="
)


print(np.nanmin(p80), np.nanmax(p80))
print(np.nanmin(p95), np.nanmax(p95))
print(np.nanmin(p99), np.nanmax(p99))
print(np.nanmin(hi))
print(np.nanmax(hi))


Loading AMJ thresholds...
Thresholds loaded.

================ 2021 ================
Loaded 91 daily layers
Max score: 19.0
Saved: SUMMER_HEATDAY_2021.tif
Valid pixels: 2266
Max score: 19.0

================ 2022 ================
Loaded 91 daily layers
Max score: 60.0
Saved: SUMMER_HEATDAY_2022.tif
Valid pixels: 2266
Max score: 60.0

================ 2023 ================
Loaded 91 daily layers
Max score: 109.0
Saved: SUMMER_HEATDAY_2023.tif
Valid pixels: 2266
Max score: 109.0

================ 2024 ================
Loaded 91 daily layers
Max score: 107.0
Saved: SUMMER_HEATDAY_2024.tif
Valid pixels: 2266
Max score: 107.0

================ 2025 ================
Loaded 91 daily layers
Max score: 28.0
Saved: SUMMER_HEATDAY_2025.tif
Valid pixels: 2266
Max score: 28.0

ALL SUMMER HEATDAY RASTERS CREATED
32.430265990899194 45.807612383759576
33.9422620337075 48.44161070786221
35.07783474678614 50.338609178466065
15.828357824925423
49.83293984101631
